# Predicting Suicide Numbers

Predicting the number of suicides in a country, year, sex and age group, from the WHO-compiled Kaggle dataset "Suicide Rates Overview 1985 to 2016".

Three regression models — multivariable linear regression, weighted k-nearest neighbours (k=3 and k=6) and a tuned regression tree — compared on a 70/30 hold-out split plus shuffled 3-, 5- and 10-fold cross-validation and a country-grouped cross-validation, scored on RMSE, against an explicit **baseline** (predict the mean).

The modelling pipeline (pre-processing, models, evaluation) lives in `src/modelling.py`; this notebook calls it and presents the results.

Data: `data/train.csv` (18,083 rows) and `data/test.csv` (9,737 rows) from the competition, which together make up the 27,820-row Kaggle dataset. The competition test set has no target column, so it is used only to generate a submission file.


In [1]:
import sys
from pathlib import Path

import pandas as pd

# The notebook runs from notebooks/, so the repo root is its parent.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from modelling import (  # noqa: E402
    FEATURES,
    TARGET,
    build_models,
    clean_data,
    comparison_table,
    cv_table,
    grouped_table,
    holdout_table,
    load_data,
    predict_test,
)

train, test = load_data(ROOT / "data")
print("train:", train.shape, "| test:", test.shape)
print("train columns:", list(train.columns))
print("missing in train:\n", train.isna().sum())


train: (18083, 12) | test: (9737, 11)
train columns: ['id', 'country', 'year', 'sex', 'age', 'suicides_no', 'population', 'country-year', 'HDI for year', ' gdp_for_year ($) ', 'gdp_per_capita ($)', 'generation']
missing in train:
 id                        0
country                   0
year                      0
sex                       0
age                       0
suicides_no               0
population                0
country-year              0
HDI for year          12467
 gdp_for_year ($)         0
gdp_per_capita ($)        0
generation                0
dtype: int64


In [2]:
# Pre-processing (same cuts as the 2021 version of this analysis):
# - drop HDI for year (69% missing in train)
# - drop rows from 2016 (incomplete recording)
# - drop Japan (male suicide counts of 3,000-8,000 heavily skewed the target)
# - keep only four features: Age, Sex, Country, Population
print("HDI missing:", train["HDI for year"].isna().mean().round(3))
print("2016 rows:", (train["year"] == 2016).sum())
print("Japan rows:", (train["country"] == "Japan").sum())
print("max suicides_no with Japan:", train["suicides_no"].max())
print("max suicides_no without Japan:", train.loc[train["country"] != "Japan", "suicides_no"].max())

X, y, clean = clean_data(train)
print("\ncleaned train:", clean.shape)
print(clean["suicides_no"].describe()[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(1))


HDI missing: 0.689
2016 rows: 120
Japan rows: 372
max suicides_no with Japan: 8948
max suicides_no without Japan: 3427

cleaned train: (17591, 6)
mean     126.2
std      309.7
min        0.0
25%        2.0
50%       19.0
75%      103.0
max     3427.0
Name: suicides_no, dtype: float64


## Models

- **Baseline** — predict the training mean for every row. Any model that cannot
  beat this has learned nothing.
- **Linear regression** — multivariable linear regression. Categorical features
  one-hot encoded, everything scaled.
- **KNN-3 / KNN-6** — weighted k-nearest neighbours.
- **Decision tree** — regression tree. The 2021 version tuned max depth and
  minimum impurity decrease by hand; here the depth and minimum leaf size are
  chosen by 5-fold cross-validation grid search, so the choice is stated, not
  guessed.


In [3]:
models, best_tree_params, best_tree_cv = build_models(X, y)
print("best tree params:", best_tree_params)
print("best tree 5-fold CV RMSE:", round(best_tree_cv, 2))


best tree params: {'m__max_depth': None, 'm__min_samples_leaf': 5}
best tree 5-fold CV RMSE: 46.47


In [4]:
# Hold-out evaluation: 70/30 split (seed fixed so it reproduces)
holdout, _split = holdout_table(models, X, y)
print(holdout.to_string())


                          RMSE        MSE      MAE     R2
Model                                                    
Baseline (mean)        304.587  92773.423  160.606 -0.000
Linear regression      192.276  36970.075  108.822  0.602
KNN-3 (weighted)        40.045   1603.622   14.003  0.983
KNN-6 (weighted)        38.738   1500.654   13.938  0.984
Decision tree (tuned)   47.332   2240.320   17.831  0.976


In [5]:
# Cross-validation on the full cleaned training set: 3-, 5- and 10-fold, RMSE.
# The folds are SHUFFLED (random_state=42): the file is sorted by country, so
# unshuffled folds would be country blocks and the numbers would not be
# comparable across fold counts.
cv = cv_table(models, X, y)
print(cv.to_string())


                       3-fold CV RMSE  5-fold CV RMSE  10-fold CV RMSE
Model                                                                 
Baseline (mean)                309.66          309.61           309.13
Linear regression              195.81          196.19           195.94
KNN-3 (weighted)                39.65           38.26            38.13
KNN-6 (weighted)                38.71           37.01            36.53
Decision tree (tuned)           48.74           46.47            44.01


In [6]:
# Grouped cross-validation: hold out whole countries (5-fold GroupKFold).
# A random split lets a model lean on neighbouring rows from the SAME country
# in adjacent years, which flatters KNN in particular. Grouping by country
# asks the harder, honest question: how well does the model predict a country
# it has never seen?
grouped = grouped_table(models, X, y, clean["country"])
print(grouped.to_string())


                       Grouped 5-fold CV RMSE
Model                                        
Baseline (mean)                        284.81
Linear regression                      220.67
KNN-3 (weighted)                       184.54
KNN-6 (weighted)                       183.36
Decision tree (tuned)                  227.28


In [7]:
# Competition test set: no target column, so generate predictions with the best
# model by 10-fold CV and write a submission file in the competition's format
best_name, submission = predict_test(models, cv, X, y, test)
submission.to_csv(ROOT / "outputs" / "submission.csv", index=False)
print("best model by 10-fold CV:", best_name)
print("submission rows:", len(submission), "| predicted mean:", round(submission["suicides_no"].mean(), 1),
      "| predicted max:", submission["suicides_no"].max())
print(submission.head().to_string(index=False))


best model by 10-fold CV: KNN-6 (weighted)
submission rows: 9737 | predicted mean: 140.7 | predicted max: 2931
   id  suicides_no
18083            0
18084           31
18085          134
18086           63
18087           92


In [8]:
# Comparison with the 2021 figures, computed from the tables above
# (no hand-copied numbers in the results themselves)
comp = comparison_table(holdout, cv, grouped)
print(comp.to_string())


                       2021 hold-out RMSE  Hold-out RMSE  2021 10-fold CV RMSE  10-fold CV RMSE  Grouped 5-fold CV RMSE
Baseline (mean)                       NaN         304.59                   NaN           309.13                  284.81
Decision tree (tuned)               65.62          47.33                230.86            44.01                  227.28
KNN-3 (weighted)                   268.85          40.04                254.85            38.13                  184.54
KNN-6 (weighted)                   241.17          38.74                241.95            36.53                  183.36
Linear regression                  239.08         192.28                242.63           195.94                  220.67


## How to read this

1. **The baseline matters.** Without it, models are only compared against each
   other. The mean-predictor baseline shows how much of the RMSE is just the
   target's spread, and every model here has to beat it to claim it learned
   anything.
2. **KNN works because the categories are one-hot encoded.** Converting them
   to integers would make nearest-neighbour distances meaningless across
   countries. Most of KNN's advantage on a random split is neighbouring rows
   from the same country in adjacent years, whose suicide counts barely
   change. Hold out whole countries instead, and KNN falls from 36.53 to
   183.36 — still the best model, but by a much smaller margin. The grouped
   CV is the honest number for "predict a country you have never seen"; the
   random-split numbers are the honest number for "fill in a missing year
   for a known country". Both are reported; neither is hidden.


## Conclusion

KNN-6 is the model to use. It is the best on every split: hold-out 38.74,
10-fold CV 36.53, grouped 183.36. The tree, tuned by cross-validated grid
search (no max depth, min leaf 5), is a credible second on random splits
but generalises worse to unseen countries than linear regression does.

The honest expectation depends on the question. For filling in a missing
year for a country already in the data, expect errors around 37 suicides.
For a country the model has never seen, expect around 183 — still the best
of the five models, but a much smaller lead. The grouped cross-validation
is the number to quote for generalisation; the random-split numbers are the
number for imputation.


## Background: the 2021 version

This analysis started as a 2021 Machine Learning assignment (R, Kaggle
InClass competition "Suicide Rates Prediction", Cyber Labs). The 2021
figures, for comparison: linear regression 239.08 (hold-out) / 242.63
(10-fold CV); KNN-3 268.85 / 254.85; KNN-6 241.17 / 241.95; decision tree
65.62 / 230.86. That version had no baseline, converted categories to
integers (which is why its KNN scored ~242 instead of ~37), and could not
submit to Kaggle (an unresolved error), so no leaderboard placement exists.
Its decision-tree hold-out figure (RMSE 65.62, R² 0.955) did not survive
its own cross-validation (230.86). Two small errors in the 2021 report are
corrected here: KNN-3's MSE was listed as 7,278.779 when 268.847² is about
72,279, and the stated maximum of 3,247 after removing Japan is 3,427 in
the data.
